In [1]:
import spacy

nlp = spacy.load("pl_core_news_lg")

In [2]:
import pandas as pd


gold_answers_train = pd.read_json("./data/convos.jsonl", lines=True)[
    "reformulated question"
].to_list()
gold_answers_val = pd.read_json("./data/rag_paraphrase_eval.jsonl", lines=True)[
    "gold_paraphrase"
].to_list()

gold_answers = [*gold_answers_train, *gold_answers_val]

In [3]:
gold_nlp_docs = [doc for doc in nlp.pipe(gold_answers)]

In [4]:
from collections.abc import Callable, Sequence
import math

import numpy as np
from spacy.ml import Doc
from spacy.tokens import Token


def is_valid_word_token(token: Token):
    if token.is_punct:
        return False

    if token.is_stop:
        return False

    if token.like_num:
        return False

    return token.pos_ in ["NOUN", "PROPN", "ADJ", "VERB", "ADV"]


def is_oov_token(token: Token):
    return token.is_oov


def select_lemmas_by(
    docs: Sequence[Doc], token_selector: Callable[[Token], bool] = is_valid_word_token
):
    lemmas_sets = get_lemmas_sets(docs, token_selector)
    lemmas_set = set().union(*lemmas_sets)

    return lemmas_set


def get_lemmas_sets(docs: Sequence[Doc], token_selector: Callable[[Token], bool]):
    lemmas_sets = [
        {token.lemma_ for token in doc if token_selector(token)} for doc in docs
    ]
    return lemmas_sets


lemmas_per_doc = get_lemmas_sets(gold_nlp_docs, token_selector=is_valid_word_token)

lemmas_df = pd.DataFrame(
    select_lemmas_by(gold_nlp_docs, is_valid_word_token), columns=["lemma"]
)
lemmas_df["df"] = lemmas_df["lemma"].apply(
    lambda l: sum(l in lemmas_set for lemmas_set in lemmas_per_doc)
)
N = len(lemmas_df)
lemmas_df["idf"] = lemmas_df["df"].apply(lambda df: math.log(N / (df + 1)))

oov_weight = np.percentile(lemmas_df["idf"], 95).item()

In [5]:
lemmas_df.set_index("lemma")["idf"]

lemma
koryguć       5.402677
azbestowy     5.402677
uciążliwy     5.402677
ślubem        4.709530
prawny        4.997212
                ...   
E             5.402677
podać         4.997212
umowa         5.402677
służebność    5.402677
inwestor      5.402677
Name: idf, Length: 444, dtype: float64

In [ ]:
from metrics.IDFWeightedTermF1 import IDFWeightedTermF1

metric = IDFWeightedTermF1(gold_questions_list=gold_answers, nlp=nlp)

metric(
    "Jak załatwić we Wrocławiu sprawę: dodatek mieszkaniowy dla gospodarstwa domowego? Chodzi mi o dokumenty, opłaty i miejsce złożenia wniosku.",
    "Jakie dokumenty i opłaty do dodatku mieszkaniowego i gdzie złożyć wniosek katastralny?",
)


(0.5193632337814724,
 'Brakujące ważne terminy: katastralny (5.52).\nNadmiarowe terminy: chodzić, domowy, gospodarstwo, miejsce (5.52), sprawa (4.82), załatwić (3.72), Wrocław (1.73).')

In [ ]:
from metrics.IDFWeightedTermF1 import ContextCarryoverRecall

metric_recall = ContextCarryoverRecall(gold_answers, nlp)
metric_recall(
    "Jak załatwić we Wrocławiu sprawę: dodatek dla gospodarstwa domowego? Chodzi mi o dokumenty, opłaty i miejsce złożenia wniosku.",
    "Jakie dokumenty i opłaty do dodatku mieszkaniowego i katastralnego?",
    "Jakie dokumenty i opłaty do tego?",
)


(0.318095366712397,
 'Parafraza nie dodaje ważnych terminów do ostatniego pytania użytkownika. Brakuje: "katastralny", "mieszkaniowy"')